# WRDS Financial Data Pull — Monthly Panel

Pulls firm-level monthly data from **Compustat** (Fundamentals Quarterly) and **CRSP** (Monthly Stock File), computes financial ratios, and builds a clean monthly panel.

Output: `wrds_monthly_financials_2006_2026.csv` with an added `ret` (delisting-adjusted monthly return) column.

In [1]:
!pip install wrds pandas numpy

## 1. Configuration

In [2]:
import wrds
import pandas as pd
import numpy as np
from datetime import datetime

START_DATE  = "2005-07-01"   # pull early to build TTM (needs 4 prior quarters) for Jan 2006
END_DATE    = datetime.today().strftime("%Y-%m-%d")
FINAL_START = "2006-01-01"   # actual start of the output panel

REPORTING_LAG_MONTHS = 3     # conservative fallback lag when rdq is missing

## 2. SIC → industry labels (4 target industries)

In [3]:
def sic_to_industry(sic):
    """Map a 4-digit SIC code to one of the 4 target industries."""
    if sic is None or (isinstance(sic, float) and np.isnan(sic)):
        return None
    sic = int(sic)
    if 6000 <= sic <= 6999:
        return "Finance"
    if (3570 <= sic <= 3579 or 3660 <= sic <= 3692 or 3694 <= sic <= 3699
            or 3810 <= sic <= 3829 or 7370 <= sic <= 7379):
        return "Technology"
    if 1200 <= sic <= 1399 or 2900 <= sic <= 2999:
        return "Energy"
    if 2830 <= sic <= 2836:
        return "Pharmaceuticals"
    return None

## 3. Connect to WRDS

In [4]:
print("Connecting to WRDS...")
db = wrds.Connection()   # prompts for credentials on first use

Connecting to WRDS...


Enter your WRDS username [Aliaksei]: ap4522
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\Aliaksei\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## 4. Pull Compustat quarterly fundamentals

In [5]:
print("Querying Compustat quarterly fundamentals...")
comp_query = f"""
    SELECT
        a.gvkey,
        a.datadate,
        a.rdq,                 -- *** report date (when data became PUBLIC) ***
        a.fyearq,
        a.fqtr,
        a.conm        AS company_name,
        a.tic         AS ticker,
        b.sic         AS sic_code,

        /* Income-statement items (quarterly) */
        a.revtq,               -- Total Revenue
        a.niq,                 -- Net Income
        a.xintq,               -- Interest Expense
        a.oibdpq,              -- Operating Income Before Depreciation
        a.dpq,                 -- Depreciation & Amortization

        /* Balance-sheet items (point-in-time) */
        a.dlttq,               -- Long-Term Debt
        a.dlcq,                -- Debt in Current Liabilities (short-term debt)
        a.seqq,                -- Stockholders' Equity – Total
        a.cshoq,               -- Common Shares Outstanding

        /* Per-share */
        a.epspxq               -- EPS (Diluted) Excluding Extraordinary Items
    FROM comp.fundq AS a
    LEFT JOIN comp.company AS b
        ON a.gvkey = b.gvkey
    WHERE a.indfmt  = 'INDL'
      AND a.datafmt = 'STD'
      AND a.popsrc  = 'D'
      AND a.consol  = 'C'
      AND a.datadate BETWEEN '{START_DATE}' AND '{END_DATE}'
      AND (
            (b.sic BETWEEN '1200' AND '1399')
         OR (b.sic BETWEEN '2900' AND '2999')
         OR (b.sic BETWEEN '2830' AND '2836')
         OR (b.sic BETWEEN '3570' AND '3579')
         OR (b.sic BETWEEN '3660' AND '3692')
         OR (b.sic BETWEEN '3694' AND '3699')
         OR (b.sic BETWEEN '3810' AND '3829')
         OR (b.sic BETWEEN '7370' AND '7379')
         OR (b.sic BETWEEN '6000' AND '6999')
      )
"""
comp = db.raw_sql(comp_query)
print(f"  -> {len(comp):,} quarterly observations pulled.")

Querying Compustat quarterly fundamentals...
  -> 620,989 quarterly observations pulled.


## 5. Pull CRSP monthly stock data — prices AND delisting-adjusted returns

In [6]:
print("Querying CRSP monthly stock file (with returns)...")
crsp_query = f"""
    SELECT
        a.permno,
        a.date,
        a.prc,
        a.ret,                 -- *** holding-period return (incl. dividends) ***
        a.shrout,
        b.gvkey
    FROM crsp.msf AS a
    INNER JOIN crsp.ccmxpf_lnkhist AS b
        ON a.permno = b.lpermno
        AND b.linktype IN ('LU','LC')
        AND b.linkprim IN ('P','C')
        AND a.date >= b.linkdt
        AND (a.date <= b.linkenddt OR b.linkenddt IS NULL)
    WHERE a.date BETWEEN '{START_DATE}' AND '{END_DATE}'
"""
crsp = db.raw_sql(crsp_query)
print(f"  -> {len(crsp):,} monthly stock observations pulled.")

print("Querying CRSP delisting returns...")
dl_query = f"""
    SELECT permno, dlstdt, dlret
    FROM crsp.msedelist
    WHERE dlstdt BETWEEN '{START_DATE}' AND '{END_DATE}'
"""
delist = db.raw_sql(dl_query)
print(f"  -> {len(delist):,} delisting events pulled.")

db.close()
print("WRDS connection closed.\n")

Querying CRSP monthly stock file (with returns)...
  -> 1,351,621 monthly stock observations pulled.
Querying CRSP delisting returns...
  -> 19,687 delisting events pulled.
WRDS connection closed.



## 6. Compute financial ratios

In [7]:
comp["datadate"] = pd.to_datetime(comp["datadate"])
comp["rdq"]      = pd.to_datetime(comp["rdq"], errors="coerce")
comp = comp.sort_values(["gvkey", "datadate"]).reset_index(drop=True)

# EBIT (quarterly) = Operating Income Before Dep. - Depreciation
comp["ebitq"] = comp["oibdpq"].fillna(0) - comp["dpq"].fillna(0)

# Trailing-twelve-month sums for FLOW variables (require a full 4 quarters)
ttm_cols = ["revtq", "niq", "xintq", "ebitq", "epspxq"]
for col in ttm_cols:
    comp[col + "_ttm"] = (
        comp.groupby("gvkey")[col]
        .rolling(window=4, min_periods=4)
        .sum()
        .reset_index(level=0, drop=True)
    )

# Interest Coverage = EBIT(TTM) / Interest Expense(TTM)
comp["interest_coverage"] = np.where(
    comp["xintq_ttm"].abs() > 0, comp["ebitq_ttm"] / comp["xintq_ttm"], np.nan)

# Total Debt / Capital (balance-sheet, point-in-time)
comp["total_debt"] = comp["dlcq"].fillna(0) + comp["dlttq"].fillna(0)
comp["capital"]    = comp["total_debt"] + comp["seqq"].fillna(0)
comp["debt_to_capital"] = np.where(
    comp["capital"].abs() > 0, comp["total_debt"] / comp["capital"], np.nan)

# Net Income Margin = Net Income(TTM) / Revenue(TTM)
comp["net_income_margin"] = np.where(
    comp["revtq_ttm"].abs() > 0, comp["niq_ttm"] / comp["revtq_ttm"], np.nan)

In [8]:
# Availability date = when the market could actually have known these numbers
comp["avail_date"] = comp["rdq"]
missing = comp["avail_date"].isna()
comp.loc[missing, "avail_date"] = (
    comp.loc[missing, "datadate"] + pd.DateOffset(months=REPORTING_LAG_MONTHS))

# Safety: rdq should never be before the fiscal period end; floor it if so
bad = comp["avail_date"] < comp["datadate"]
comp.loc[bad, "avail_date"] = (
    comp.loc[bad, "datadate"] + pd.DateOffset(months=REPORTING_LAG_MONTHS))

# Diagnostic: how large is the reporting lag we are applying?
lag_days = (comp["avail_date"] - comp["datadate"]).dt.days
print("Reporting lag applied (days):")
print(lag_days.describe().round(1))
print(f"\nRows using actual rdq : {(~missing).sum():,}")
print(f"Rows using +{REPORTING_LAG_MONTHS}mo fallback : {missing.sum():,}")

# Map each quarter into the month it became AVAILABLE (not the fiscal month)
comp["ym"] = comp["avail_date"].dt.to_period("M")

# If two filings map to the same availability month for one firm, keep the latest fiscal one
comp = comp.sort_values(["gvkey", "datadate"]).drop_duplicates(
    subset=["gvkey", "ym"], keep="last")

Reporting lag applied (days):
count    620989.0
mean         67.9
std          40.7
min           0.0
25%          38.0
50%          88.0
75%          91.0
max        2450.0
dtype: float64

Rows using actual rdq : 336,057
Rows using +3mo fallback : 284,932


## 7. Build survivorship-free monthly returns from CRSP

Combine the normal monthly return with the delisting return so delisted firms contribute their final return. The combined return for a delisting month is `(1+ret)(1+dlret) - 1`; where `ret` is missing in the delisting month, the delisting return stands alone.

In [9]:
crsp["date"] = pd.to_datetime(crsp["date"])
crsp["prc"]  = crsp["prc"].abs()             # CRSP stores negative prc for bid/ask midpoints
crsp["ym"]   = crsp["date"].dt.to_period("M")

# Attach delisting returns by permno + month
delist["dlstdt"] = pd.to_datetime(delist["dlstdt"])
delist["ym"]     = delist["dlstdt"].dt.to_period("M")
delist = delist.drop_duplicates(subset=["permno", "ym"], keep="last")

crsp = crsp.merge(delist[["permno", "ym", "dlret"]], on=["permno", "ym"], how="left")

# Delisting-adjusted monthly return
crsp["ret_adj"] = np.where(
    crsp["dlret"].notna() & crsp["ret"].notna(),
    (1 + crsp["ret"]) * (1 + crsp["dlret"]) - 1,
    crsp["ret"].fillna(crsp["dlret"]),      # use whichever exists
)

# One row per gvkey-month (keep last if multiple permnos map via the link)
crsp = crsp.sort_values(["gvkey", "date"]).drop_duplicates(
    subset=["gvkey", "ym"], keep="last")

## 8. Build the monthly panel and forward-fill from the availability month

In [10]:
print("Building monthly panel (forward-filling from availability date)...")

firm_bounds = comp.groupby("gvkey")["avail_date"].agg(["min"]).reset_index()
end_period  = pd.Timestamp(END_DATE).to_period("M")

spine_parts = []
for _, r in firm_bounds.iterrows():
    periods = pd.period_range(r["min"].to_period("M"), end_period, freq="M")
    spine_parts.append(pd.DataFrame({"gvkey": r["gvkey"], "ym": periods}))
spine = pd.concat(spine_parts, ignore_index=True)

panel = spine.merge(comp, on=["gvkey", "ym"], how="left").sort_values(["gvkey", "ym"])

# Forward-fill fundamentals within each firm (carries last PUBLISHED data forward)
ff_cols = [c for c in panel.columns if c not in ("gvkey", "ym")]
panel[ff_cols] = panel.groupby("gvkey")[ff_cols].ffill()

panel = panel.dropna(subset=["company_name"])
panel["month"] = panel["ym"].dt.to_timestamp()

Building monthly panel (forward-filling from availability date)...


## 9. Merge CRSP price + returns and compute P/E

In [11]:
panel = panel.merge(
    crsp[["gvkey", "ym", "prc", "ret_adj"]], on=["gvkey", "ym"], how="left")

panel["pe_ratio"] = np.where(
    panel["epspxq_ttm"].abs() > 0, panel["prc"] / panel["epspxq_ttm"], np.nan)

## 10. Add industry labels, trim, and select output columns

In [12]:
panel["sic_code"] = pd.to_numeric(panel["sic_code"], errors="coerce")
panel["industry"] = panel["sic_code"].apply(sic_to_industry)
panel = panel[panel["industry"].notna()].copy()

panel = panel[panel["month"] >= FINAL_START].copy()

# Expose the delisting-adjusted CRSP return (ret_adj) as 
output = panel[[
    "gvkey", "ticker", "company_name", "sic_code", "industry", "month",
    "interest_coverage", "debt_to_capital", "pe_ratio", "net_income_margin",
    "ret_adj",
]].rename(columns={"month": "date", "ret_adj": "ret"})

output = output.sort_values(["gvkey", "date"]).reset_index(drop=True)

## 11. Save

In [13]:
OUT_FILE = "wrds_monthly_financials_2006_2026.csv"
output.to_csv(OUT_FILE, index=False)

print(f"Done!  {len(output):,} rows x {len(output.columns)} columns")
print(f"Unique firms: {output['gvkey'].nunique():,}")
print(f"Date range:   {output['date'].min()} -> {output['date'].max()}")
print(f"Saved to:     {OUT_FILE}")
print("""
Columns:
  gvkey, ticker, company_name, sic_code, industry, date
  interest_coverage  - EBIT(TTM) / Interest Expense(TTM)
  debt_to_capital    - (ST + LT Debt) / (Total Debt + Equity)
  pe_ratio           - Price / Diluted EPS (TTM)
  net_income_margin  - Net Income(TTM) / Revenue(TTM)
  ret                - delisting-adjusted CRSP monthly return (survivorship-free)

Fundamentals are now stamped by AVAILABILITY DATE (rdq, or datadate+3mo),
so ranking on month t-1 no longer uses data that was not yet public.
Use the `ret` column in the strategy notebook instead of yfinance.""")

Done!  2,967,224 rows x 11 columns
Unique firms: 20,291
Date range:   2006-01-01 00:00:00 -> 2026-10-01 00:00:00
Saved to:     wrds_monthly_financials_2006_2026.csv

Columns:
  gvkey, ticker, company_name, sic_code, industry, date
  interest_coverage  - EBIT(TTM) / Interest Expense(TTM)
  debt_to_capital    - (ST + LT Debt) / (Total Debt + Equity)
  pe_ratio           - Price / Diluted EPS (TTM)
  net_income_margin  - Net Income(TTM) / Revenue(TTM)
  ret                - delisting-adjusted CRSP monthly return (survivorship-free)

Fundamentals are now stamped by AVAILABILITY DATE (rdq, or datadate+3mo),
so ranking on month t-1 no longer uses data that was not yet public.
Use the `ret` column in the strategy notebook instead of yfinance.
